# Class 7 — The logic of pandas

The goal today is not to memorise pandas syntax (the documentation does that better than any class).
The goal is to understand **how pandas thinks about data**, because that is what makes it powerful:

1. A table is a collection of **typed columns**, not a list of rows → operations apply to whole columns at once (*vectorisation*).
2. A **boolean column** is a question you ask the data → filtering.
3. Every row has a **label** (the *index*) → pandas matches data by label, not by position (*alignment*).
4. **Split–apply–combine** (`groupby`) → most descriptive statistics in one line.
5. The **shape** of a table (*tidy* vs *wide*) decides which operations are easy.

Parts 1 and 2 are guided. In Part 3 you work on your own with a messier, real dataset.

> Run the cells in order with **Shift+Enter**. If something breaks, *Restart* the kernel and *Run All*.

In [ ]:
import pandas as pd
pd.__version__

---
# Part 1 — How pandas thinks

## 1.1 From a list of rows to a table of columns

Until now we stored records as a list of dictionaries — one dictionary per observation:

In [ ]:
records = [
    {'month': 'aug', 'temp': 22.2, 'RH': 29, 'area': 0.0},
    {'month': 'aug', 'temp': 24.1, 'RH': 27, 'area': 0.0},
    {'month': 'sep', 'temp': 13.1, 'RH': 63, 'area': 3.5},
    {'month': 'mar', 'temp':  8.2, 'RH': 51, 'area': 0.0},
]
df = pd.DataFrame(records)
df

pandas reorganised the rows into **columns**. Each column has a single data type:

In [ ]:
df.dtypes

A column on its own is a `Series`: an array of values **plus** an index (the labels 0, 1, 2, 3 on the left).
A `DataFrame` is a set of Series that share the same index — you can think of it as a dictionary of columns.

In [ ]:
s = df['temp']
print(type(s))
print(s.values)   # the values
print(s.index)    # the labels

## 1.2 Vectorisation: describe *what*, not *how*

Because a column has one type, pandas can apply an operation to the whole column at once.
Compare with the loops we wrote in previous classes:

In [ ]:
# the "Python lists" way
temps_f = []
for t in df['temp']:
    temps_f.append(t * 9 / 5 + 32)
temps_f

In [ ]:
# the pandas way: an expression on the whole column
df['temp_f'] = df['temp'] * 9 / 5 + 32
df

It is not only shorter: it is much faster, because the loop runs in compiled code instead of in Python.
Let's check with a table of half a million rows.

In [ ]:
import time

big = pd.DataFrame({'temp': list(df['temp']) * 125_000})
print(len(big), 'rows')

start = time.perf_counter()
result = []
for t in big['temp']:
    result.append(t * 9 / 5 + 32)
t_loop = time.perf_counter() - start

start = time.perf_counter()
result = big['temp'] * 9 / 5 + 32
t_vec = time.perf_counter() - start

print(f'loop: {t_loop:.3f} s   vectorised: {t_vec:.4f} s   -> {t_loop / t_vec:.0f}x faster')

## 1.3 The forest fires dataset

517 fires in the Montesinho natural park (Trás-os-Montes), with weather conditions and burned area
([source](https://archive.ics.uci.edu/dataset/162/forest+fires)).

| column | meaning |
|---|---|
| `X`, `Y` | cell in a 9×9 grid over the park map |
| `month`, `day` | month and weekday (no year!) |
| `FFMC`, `DMC`, `DC`, `ISI` | indices of the Canadian Fire Weather Index (FWI) system |
| `temp`, `RH`, `wind`, `rain` | temperature (°C), relative humidity (%), wind (km/h), rain (mm) |
| `area` | burned area (ha) |

We read it directly from the course repository, so there is nothing to download.

In [ ]:
URL = 'https://raw.githubusercontent.com/isa-ulisboa/greends-ipython/main/datasets/'
fires = pd.read_csv(URL + 'forestfires.csv')
fires.head()

In [ ]:
fires.shape

In [ ]:
fires.info()

In [ ]:
fires.describe()

**Look before you compute.** In the `area` column the median (50%) is about 0.5 ha but the maximum is 1090 ha.
What fraction of the fires burned (almost) nothing?

In [ ]:
(fires['area'] == 0).mean()

That one line uses two of today's ideas: `fires['area'] == 0` is a *vectorised comparison* that produces a
column of `True`/`False`, and the mean of booleans is the proportion of `True` (since `True` counts as 1).
So the *average* burned area is a poor summary of a "typical" fire here:

In [ ]:
fires['area'].mean(), fires['area'].median()

## 1.4 Boolean masks: asking questions to the data

A comparison on a column gives a boolean Series of the same length — a *mask*.
Using a mask inside `[...]` keeps the rows where it is `True`.

In [ ]:
mask = fires['temp'] > 25
mask.head(10)

In [ ]:
print(mask.sum(), 'fires with temp > 25')
fires[mask]

Masks are combined with `&` (and), `|` (or) and `~` (not). Each condition needs its own parentheses.
`.isin()` checks membership in a list.

In [ ]:
summer = fires['month'].isin(['jun', 'jul', 'aug', 'sep'])
dry = fires['RH'] < 30
fires[summer & dry & (fires['area'] > 10)]

**Exercise 1.4** — How many fires happened on weekends (`sat`, `sun`)? What percentage of all fires is that?
Is it more than you would expect if fires were spread evenly over the week?

In [ ]:
weekend = fires['day'].isin(['sat', 'sun'])
print(weekend.sum(), 'fires on weekends')
print(f'{weekend.mean():.1%} of fires, versus {2/7:.1%} expected')

## 1.5 The index: labels, not positions

Every row has a **label** (the index) and a **position** (0, 1, 2, …). When you read a CSV, pandas uses
labels 0, 1, 2, … so **label and position coincide**. This is why `loc` (by label) and `iloc` (by position)
often seem to do the same thing. They are not the same — and the difference appears as soon as we filter:

In [ ]:
hot = fires[fires['temp'] > 25]
hot.head(3)          # look at the labels on the left: the original row labels are kept

In [ ]:
hot.iloc[0]          # the first row of `hot`, by POSITION

In [ ]:
try:
    hot.loc[0]       # the row with LABEL 0 -- it was filtered out!
except KeyError as e:
    print('KeyError:', e)

Keeping the original labels is a feature: each row in `hot` still "knows" which fire it is.

Labels do not have to be numbers. With `set_index` a column becomes the index:

In [ ]:
months = pd.DataFrame({
    'mth':  ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'],
    'name': ['January', 'February', 'March', 'April', 'May', 'June',
             'July', 'August', 'September', 'October', 'November', 'December'],
    'season': ['winter', 'winter', 'spring', 'spring', 'spring', 'summer',
               'summer', 'summer', 'autumn', 'autumn', 'autumn', 'winter'],
})
months = months.set_index('mth')
months

In [ ]:
print(months.loc['aug', 'name'])     # by label
print(months.iloc[7, 0])             # by position: row 7, column 0
months.loc['jun':'aug']              # label slices INCLUDE the end label

In [ ]:
months.iloc[5:8]                     # position slices EXCLUDE the end, like Python lists

### Why the index matters: alignment

When pandas combines two Series, it **matches values by label**, not by position.

In [ ]:
a = pd.Series([1, 2, 3], index=['jan', 'feb', 'mar'])
b = pd.Series([10, 20, 30], index=['mar', 'jan', 'feb'])
a + b

In [ ]:
c = pd.Series([100, 200], index=['jan', 'apr'])
a + c                                # no partner -> NaN (missing), never a silent mismatch

A realistic case: the number of fires per month in two subsets of the data. The two results have different
months in a different order, yet the division is correct because it is done label by label.

In [ ]:
all_per_month = fires['month'].value_counts()
big_per_month = fires.loc[fires['area'] > 10, 'month'].value_counts()
share_big = (big_per_month / all_per_month).fillna(0)
share_big.sort_values(ascending=False)

With lists we would have to make sure, by hand, that position *i* in one list corresponds to position *i*
in the other. With an index, pandas does it for us.

**Summary**

| | `loc` | `iloc` |
|---|---|---|
| selects by | label | position |
| slice `a:b` | includes `b` | excludes `b` |
| slice example | `fires.loc[0:4, 'temp':'wind']` | `fires.iloc[0:4, 8:10]` |
| typical use | `fires.loc[mask, ['temp', 'RH']]` | `fires.iloc[:20, 4:8]` |

Compare the two slices: same numbers, different results.

In [ ]:
display(fires.loc[0:4, 'temp':'wind'])    # labels 0..4 and columns temp..wind: both ends included
display(fires.iloc[0:4, 8:10])            # positions 0..3 and columns 8, 9: end excluded

**Exercise 1.5** — Using `loc`, select the FWI variables (`FFMC`, `DMC`, `DC`, `ISI`) of the fires in August
and September. Then, using `iloc`, take the first 5 rows of that result. Why does `.loc[0:4]` not give the same result?

In [ ]:
aug_sep = fires.loc[fires['month'].isin(['aug', 'sep']), ['FFMC', 'DMC', 'DC', 'ISI']]
display(aug_sep.iloc[0:5])
display(aug_sep.loc[0:4])   # labels 0..4 -- only those labels that survived the filter

---
# Part 2 — Split, apply, combine; tidy and wide data

## 2.1 `groupby`

A very common question: *for each group, compute a summary*. With what we knew before, we would write:

In [ ]:
count = {}
for m in fires['month']:
    count[m] = count.get(m, 0) + 1
count

`groupby` does the three steps for us: **split** the rows by month, **apply** a function to each group,
**combine** the results into a new table.

In [ ]:
fires.groupby('month')['area'].size()

In [ ]:
by_month = fires.groupby('month').agg(
    n_fires=('area', 'size'),
    avg_temp=('temp', 'mean'),
    avg_RH=('RH', 'mean'),
    total_area=('area', 'sum'),
)
by_month

Notice that the group key (`month`) became the **index** of the result. Two consequences:

1. The months are sorted alphabetically (`apr`, `aug`, `dec`, …). Since the index holds labels, we can put the rows
   in calendar order with `reindex`.
2. If we need `month` as an ordinary column again (e.g. to export or merge), we use `reset_index()`.

`reindex(labels)` returns a new Series (or DataFrame) with **exactly these labels, in this order**:
existing labels keep their values, new labels get `NaN`, and labels that are not in the list are dropped.
With the Series `a` from section 1.5:

In [ ]:
a.reindex(['mar', 'jan', 'apr'])     # mar and jan reordered, apr is new (NaN), feb is dropped

In [ ]:
# months.index holds the labels 'jan', 'feb', ..., 'dec', in calendar order
by_month = by_month.reindex(months.index)   # missing months would appear as NaN
by_month

In [ ]:
by_month.reset_index().head(3)

**Exercise 2.1** — Group by `day` instead. On which weekday is the *median* burned area largest?
And the number of fires?

In [ ]:
by_day = fires.groupby('day')['area'].agg(['size', 'median'])
by_day.sort_values('median', ascending=False)

## 2.2 Joining tables: `merge`

Information often lives in different tables. `merge` combines them using a common key
(like a database join). Here we add the full month name and the season to each fire.

In [ ]:
fires2 = fires.merge(months, left_on='month', right_index=True, how='left')
fires2[['month', 'name', 'season', 'area']].head()

In [ ]:
fires2.groupby('season')['area'].agg(['size', 'sum', 'median'])

## 2.3 Tidy vs wide data

A table is **tidy** when

- each **variable** is a column,
- each **observation** is a row,
- each type of observational unit is a table.

`fires` is tidy: one row per fire, one column per variable. Now let's build a table of total burned area by
month and weekday:

In [ ]:
wide = fires.pivot_table(index='month', columns='day', values='area', aggfunc='sum', fill_value=0)
wide = wide.reindex(index=months.index, columns=['mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun'])
wide.round(0)

This is a **wide** table: the weekday, which is a *variable*, has been spread over 7 columns.
Wide tables are good for **people** (easy to read, compare and print).
Tidy tables are good for **computing**: filtering, grouping, merging and plotting all expect one variable per column.

`melt` turns a wide table back into a tidy (long) one:

In [ ]:
long = wide.reset_index().melt(id_vars='mth', var_name='day', value_name='area')
long

In [ ]:
# questions about "day" are now ordinary operations on a column
long[long['day'].isin(['sat', 'sun'])].groupby('mth')['area'].sum().reindex(months.index)

**Rule of thumb:** keep your data tidy while you work; make it wide at the end, for a report.

Many official statistics (INE, Eurostat, …) are published *wide*, for people. That is exactly what you will
deal with in Part 3.

---
# Part 3 — On your own: rural fires in Portugal (INE)

The file `rural_fires.csv` was downloaded from [INE](https://www.ine.pt): number of rural fires in 2023 by
geographic unit (country, NUTS regions, municipalities) and cause of fire.

Real data are rarely clean. Before using pandas, **look at the raw file**: read it as plain text (as in Class 6)
and print the first 15 lines.

In [ ]:
import urllib.request

with urllib.request.urlopen(URL + 'rural_fires.csv') as f:
    lines = f.read().decode('utf-8').splitlines()
for i, line in enumerate(lines[:15]):
    print(i, line)

Things to notice:

- there are 11 lines of titles before the data, and notes at the end of the file;
- the separator is `;`, not `,`;
- every number is followed by ` &` (a flag added by INE), so pandas will read the numbers as text;
- the first column mixes a **code** and a **name** (`1111601: Arcos de Valdevez`);
- the rows mix different units: the country, NUTS regions and municipalities.

The following cell reads only the 347 data rows and gives the columns clean names.

In [ ]:
causes = ['negligence', 'intentional', 'natural', 'reignition', 'undetermined', 'not_investigated']
rf = pd.read_csv(URL + 'rural_fires.csv', sep=';', skiprows=11, nrows=347, header=None,
                 usecols=range(8), names=['place', 'total'] + causes)
rf.head()

The exercises go from basic to challenging. Do as many as you can.

### Basic

**3.1** Check the `dtypes` of `rf`. Convert the numeric columns to integers: remove the `&`, strip the spaces,
and use `.astype(int)`. (Hint: string methods on a whole column are available through `.str`, e.g. `rf['total'].str.replace(...)`.)

In [ ]:
for col in ['total'] + causes:
    rf[col] = rf[col].str.replace('&', '').str.strip().astype(int)
rf.dtypes

**3.2** Split `place` into two new columns, `code` and `name`. (Hint: `.str.split(': ', n=1, expand=True)`.)

In [ ]:
rf[['code', 'name']] = rf['place'].str.split(': ', n=1, expand=True)
rf[['code', 'name', 'total']].head()

**3.3** The length of the code tells the level: `PT` is the country, 1 character is NUTS I, 2 characters
NUTS II, 3 characters NUTS III and 7 characters a municipality. Create a dataframe `mun` with the
municipalities only. How many are there?

In [ ]:
mun = rf[rf['code'].str.len() == 7]
len(mun)

**3.4** Which 10 municipalities had the most fires? (`sort_values` or `nlargest`.)

In [ ]:
mun.nlargest(10, 'total')[['name', 'total']]

### Intermediate

**3.5** *Validate the data*: does the sum of `total` over the municipalities equal the value for Portugal?
And for each municipality, is `total` equal to the sum of the six causes?

In [ ]:
print(mun['total'].sum(), rf.loc[rf['code'] == 'PT', 'total'].item())
print((mun[causes].sum(axis=1) == mun['total']).all())

**3.6** Add a column `pct_negligence` with the percentage of fires due to negligence. Which municipalities
with at least 50 fires have the highest percentage?

In [ ]:
mun = mun.copy()
mun['pct_negligence'] = 100 * mun['negligence'] / mun['total']
mun[mun['total'] >= 50].sort_values('pct_negligence', ascending=False)[['name', 'total', 'pct_negligence']].head(10)

### Challenge

**3.7** The table is *wide*: the cause of fire is a variable spread over six columns. Use `melt` to create a
tidy table `tidy` with columns `code`, `name`, `cause`, `n_fires` (one row per municipality and cause).
How many rows should it have?

In [ ]:
tidy = mun.melt(id_vars=['code', 'name'], value_vars=causes, var_name='cause', value_name='n_fires')
print(len(tidy), '=', len(mun), 'x', len(causes))
tidy.head()

**3.8** Codes are hierarchical: the first 2 characters of a municipality code are its NUTS II region
(`1111601` → `11`, Norte). Add a column `nuts2` to `tidy`, and bring in the region **name** with a `merge`
with the NUTS II rows of `rf`.

In [ ]:
tidy['nuts2'] = tidy['code'].str[:2]
nuts2 = rf.loc[rf['code'].str.len() == 2, ['code', 'name']]
nuts2 = nuts2[nuts2['code'] != 'PT'].rename(columns={'code': 'nuts2', 'name': 'region'})
tidy = tidy.merge(nuts2, on='nuts2', how='left')
tidy.head()

**3.9** With `groupby` on the tidy table, compute the number of fires per region and cause. Then, for the final
report, turn it into a *wide* table (regions × causes) with the **percentage** of each cause in each region.
Which region has the largest share of intentional fires?

In [ ]:
per_region = tidy.groupby(['region', 'cause'])['n_fires'].sum().reset_index()
report = per_region.pivot_table(index='region', columns='cause', values='n_fires')
report = 100 * report.div(report.sum(axis=1), axis=0)
report.round(1).sort_values('intentional', ascending=False)

**3.10** Save your tidy table to `rural_fires_2023_tidy.csv` with `.to_csv(..., index=False)`.
Why `index=False`?

In [ ]:
tidy.to_csv('rural_fires_2023_tidy.csv', index=False)